Downloading the data.

In [ ]:
!kaggle datasets download -d rm1000/brain-tumor-mri-scans

Extracting the data.

In [ ]:
import zipfile

zip_file_path = '/content/brain-tumor-mri-scans.zip'
data_path = '/content/data'
with zipfile.ZipFile(zip_file_path, 'r') as zip_file:
  print('Unzipping_data...')
  zip_file.extractall(data_path)

In [ ]:
!pip install split-folders

Splitting the dataset in train, validation and test sets.

In [ ]:
import splitfolders

input_folder = '/content/data'
output_folder = '/content/datasets'
splitfolders.ratio(input_folder,
                   output=output_folder,
                   seed=42,
                   ratio=(0.8, 0.1, 0.1),
                   group_prefix=None,
                   move=False)

Splitting further the training set to use 70% of the initial training set to tune the hyperparameters.

In [ ]:
train_data_path = '/content/datasets/train'
tune_data = output_folder + '/tune_data'
splitfolders.ratio(train_data_path,
                    tune_data,
                    seed=42,
                    ratio=(0.7, 0.15, 0.15),
                    group_prefix=None,
                    move=False)

Validating the results of the first split.

In [ ]:
import os

total_images = 0
for roots,dirs,files in os.walk(output_folder,topdown=True):
  print(f'There are {len(dirs)} directories and {len(files)} images in {roots}')
  total_images += len(files)

print(f'There are {total_images} images in total.')

Validating the result of the second split

In [ ]:
import os

total_images = 0
for roots,dirs,files in os.walk(tune_data,topdown=True):
  print(f'There are {len(dirs)} directories and {len(files)} images in {roots}')
  total_images += len(files)

print(f'There are {total_images} images in total.')

Setting up device agnostic code

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

Defining the directories path.

In [ ]:
train_dir = '/content/datasets/train'
tune_dir = '/content/datasets/tune_data/train'
val_dir = '/content/datasets/val'
test_dir = '/content/datasets/test'

train_dir, tune_dir, val_dir, test_dir

Defining a funtion to calculate the mean and the std of the training set distribution for the normalization transformation

In [ ]:
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader

def calculate_mean_std(data_dir: str,
                       data_transforms: transforms.Compose=None):

  print(data_transforms)

  data = ImageFolder(root=data_dir,
                     transform=data_transforms,
                     target_transform=None)

  num_of_pixels = len(data) * 224 * 224

  dataloader = DataLoader(dataset=data,
                          batch_size=1000,
                          num_workers=os.cpu_count())

  total_sum = 0
  for batch in dataloader:
    total_sum += batch[0].sum()
  mean = total_sum / num_of_pixels

  sum_of_squared_error = 0
  for batch in dataloader:
    sum_of_squared_error += ((batch[0] - mean).pow(2)).sum()
  std = torch.sqrt(sum_of_squared_error / num_of_pixels)

  return mean.item(), std.item()

Retrieving the mean and std of the training set

In [ ]:
from torchvision import transforms

temp_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor()
])

mean, std = calculate_mean_std(data_dir=test_dir,
                               data_transforms=temp_transforms)

mean, std

Defining the preprocessing steps for each dataset.

In [ ]:
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(num_output_channels=1),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=45),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomAffine(degrees=0, scale=(0.8, 1.2)),
    transforms.Lambda(lambda img: transforms.functional.equalize(img)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

validation_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(num_output_channels=1),
    transforms.Lambda(lambda img: transforms.functional.equalize(img)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

test_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(num_output_channels=1),
    transforms.Lambda(lambda img: transforms.functional.equalize(img)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

Creating Pytorch Datasets

In [ ]:
from torchvision.datasets import ImageFolder

train_data = ImageFolder(root=train_dir,
                         transform=train_transforms,
                         target_transform=None)

tune_data = ImageFolder(root=tune_dir,
                        transform=train_transforms,
                        target_transform=None)

validation_data = ImageFolder(root=val_dir,
                              transform=validation_transforms,
                              target_transform=None)

test_data = ImageFolder(root=test_dir,
                        transform=test_transforms,
                        target_transform=None)

train_data, tune_data, validation_data, test_data

Defining the functions to load the data into dataloaders.

In [ ]:
from torch.utils.data import DataLoader

def train_data_loader(train_dataset: ImageFolder,
                      batch_size: int=32,
                      shuffle: bool=True,
                      num_workers: int=1):

  train_dataloader = DataLoader(dataset=train_dataset,
                                batch_size=batch_size,
                                shuffle=shuffle,
                                num_workers=num_workers)

  return train_dataloader

def tune_data_loader(tune_dataset: ImageFolder,
                     batch_size: int=32,
                     shuffle: bool=True,
                     num_workers: int=1):

  tune_dataloader = DataLoader(dataset=tune_dataset,
                                batch_size=batch_size,
                                shuffle=shuffle,
                                num_workers=num_workers)

  return tune_dataloader

def validation_data_loader(validation_dataset: ImageFolder,
                          batch_size: int=32,
                          shuffle: bool=False,
                          num_workers: int=1):

  validation_dataloader = DataLoader(dataset=validation_dataset,
                                    batch_size=batch_size,
                                    shuffle=shuffle,
                                    num_workers=num_workers)

  return validation_dataloader

def test_data_loader(test_dataset: ImageFolder,
                    batch_size: int=32,
                    shuffle: bool=False,
                    num_workers: int=1):

  test_dataloader = DataLoader(dataset=test_dataset,
                                batch_size=batch_size,
                                shuffle=shuffle,
                                num_workers=num_workers)

  return test_dataloader

Loading the data into data loaders.

In [ ]:
import os

BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()

train_dataloader = train_data_loader(train_dataset=train_data,
                                    batch_size=BATCH_SIZE,
                                    shuffle=True,
                                    num_workers=NUM_WORKERS)

tune_dataloader = tune_data_loader(tune_dataset=tune_data,
                                   batch_size=BATCH_SIZE,
                                   shuffle=True,
                                   num_workers=NUM_WORKERS)

validation_dataloader = validation_data_loader(validation_dataset=validation_data,
                                    batch_size=BATCH_SIZE,
                                    shuffle=False,
                                    num_workers=NUM_WORKERS)

test_dataloader = test_data_loader(test_dataset=test_data,
                                    batch_size=BATCH_SIZE,
                                    shuffle=False,
                                    num_workers=NUM_WORKERS)

train_dataloader, tune_dataloader, validation_dataloader, test_dataloader

Defining the model architecture.

In [ ]:
import torch
from torch import nn

class ModelArchitecture(nn.Module):
  def __init__(self, hidden_size_1, hidden_size_2, hidden_size_3, hidden_size_4):
    super().__init__()

    self.block_1 = nn.Sequential(
        nn.Conv2d(in_channels=1,
                  out_channels=32,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     stride=2)
    )

    self.block_2 = nn.Sequential(
        nn.Conv2d(in_channels=32,
                  out_channels=64,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.Conv2d(in_channels=64,
                  out_channels=128,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     stride=2)
    )

    self.block_3 = nn.Sequential(
        nn.Conv2d(in_channels=128,
                  out_channels=128,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.Conv2d(in_channels=128,
                  out_channels=128,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.Conv2d(in_channels=128,
                  out_channels=256,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     stride=2)
    )

    self.block_4 = nn.Sequential(
        nn.Conv2d(in_channels=256,
                  out_channels=256,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.Conv2d(in_channels=256,
                  out_channels=256,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.Conv2d(in_channels=256,
                  out_channels=256,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.Conv2d(in_channels=256,
                  out_channels=256,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,
                     stride=2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=256*14*14,
                  out_features=hidden_size_1),
        nn.ReLU(),
        nn.Linear(in_features=hidden_size_1,
                  out_features=hidden_size_2),
        nn.ReLU(),
        nn.Linear(in_features=hidden_size_2,
                  out_features=hidden_size_3),
        nn.ReLU(),
        nn.Linear(in_features=hidden_size_3,
                  out_features=hidden_size_4),
        nn.ReLU(),
        nn.Linear(in_features=hidden_size_4,
                  out_features=4)
    )

  def forward(self, x:torch.Tensor) -> torch.Tensor:
    return self.classifier(self.block_4(self.block_3(self.block_2(self.block_1(x)))))

Defining the train step.

In [ ]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device=device):

  model.train()

  train_loss = 0
  train_accuracy = 0

  for batch, (X_train, y_train) in enumerate(dataloader):

    X_train, y_train = X_train.to(device), y_train.to(device)

    y_logits = model(X_train)

    loss = loss_fn(y_logits, y_train)
    train_loss += loss.item()

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    y_probs = torch.softmax(y_logits, dim=1)
    y_preds = torch.argmax(y_probs, dim=1)
    accuracy = (y_preds == y_train).sum().item()/len(y_preds)
    train_accuracy += accuracy

  train_loss /= len(dataloader)
  train_accuracy /= len(dataloader)

  return train_loss, train_accuracy

Defining the validation step.

In [ ]:
def validation_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device=device):

  model.eval()

  validation_loss = 0
  validation_accuracy = 0

  with torch.inference_mode():
    for batch, (X_val, y_val) in enumerate(dataloader):

      X_val, y_val = X_val.to(device), y_val.to(device)

      validation_logits = model(X_val)

      loss = loss_fn(validation_logits, y_val)
      validation_loss += loss.item()

      validation_probs = torch.softmax(validation_logits, dim=1)
      validation_preds = torch.argmax(validation_probs, dim=1)

      accuracy = (validation_preds == y_val).sum().item()/len(validation_preds)
      validation_accuracy += accuracy

  validation_loss /= len(dataloader)
  validation_accuracy /= len(dataloader)

  return validation_loss, validation_accuracy

Defining the Early stopping class.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, delta=0, restore_best_weights=True):
        self.patience = patience
        self.delta = delta
        self.restore_best_weights = restore_best_weights
        self.best_score = None
        self.early_stop = False
        self.counter = 0
        self.best_model_state = None

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.best_model_state = model.state_dict()
        elif score < self.best_score + self.delta:  #<=
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                print(f'No improvement found in the last {self.counter} epochs. \nEarly stopping triggered.')
                if self.restore_best_weights:
                  model.load_state_dict(self.best_model_state)
                  print('Restored best weights.')
        else:
            self.best_score = score
            self.best_model_state = model.state_dict()
            self.counter = 0

Defining the training loop.

In [ ]:
from tqdm.auto import tqdm

def training_loop(model: torch.nn.Module,
                  train_dataloader: torch.utils.data.DataLoader,
                  validation_dataloader: torch.utils.data.DataLoader,
                  loss_fn: torch.nn.Module,
                  optimizer: torch.optim.Optimizer,
                  device=device,
                  early_stopping=None,
                  epochs: int=1):

  results = {'Train_loss': [],
             'Train_accuracy': [],
             'Validation_loss': [],
             'Validation_accuracy': []}

  for epoch in tqdm(range(epochs)):
    train_loss, train_accuracy = train_step(model=model,
                                            dataloader=train_dataloader,
                                            loss_fn=loss_fn,
                                            optimizer=optimizer,
                                            device=device)

    validation_loss, validation_accuracy = validation_step(model=model,
                                                          dataloader=validation_dataloader,
                                                          loss_fn=loss_fn,
                                                          device=device)

    print(
            f"Epoch: {epoch+1} | "
            f"Train_loss: {train_loss:.3f} | "
            f"Train_accuracy: {train_accuracy:.3f} | "
            f"Validation_loss: {validation_loss:.3f} | "
            f"Validation_accuracy: {validation_accuracy:.3f}"
        )

    results["Train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
    results["Train_accuracy"].append(train_accuracy.item() if isinstance(train_accuracy, torch.Tensor) else train_accuracy)
    results["Validation_loss"].append(validation_loss.item() if isinstance(validation_loss, torch.Tensor) else validation_loss)
    results["Validation_accuracy"].append(validation_accuracy.item() if isinstance(validation_accuracy, torch.Tensor) else validation_accuracy)

    early_stopping(validation_loss, model)
    if early_stopping.early_stop:
        break

  return results

In [ ]:
!pip install optuna

Defining the hyperparameters optimization function.

In [ ]:
import optuna

def hyperparameters_tuning(trial):

    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    batch_size = trial.suggest_categorical("batch_size", [2**x for x in range(1,9)])
    hidden_size_1 = trial.suggest_int('hidden_size_1', 2049, 4096)
    hidden_size_2 = trial.suggest_int('hidden_size_2', 1025, 2048)
    hidden_size_3 = trial.suggest_int('hidden_size_3', 513, 1024)
    hidden_size_4 = trial.suggest_int('hidden_size_4', 16, 512)

    train_dataloader_optim = tune_data_loader(tune_dataset=tune_data,
                                        batch_size=batch_size,
                                        shuffle=True,
                                        num_workers=NUM_WORKERS)

    validation_dataloader_optim = validation_data_loader(validation_dataset=validation_data,
    batch_size=batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS)

    torch.manual_seed(42)
    torch.cuda.manual_seed(42)

    NUM_EPOCHS = 1000

    model = ModelArchitecture(hidden_size_1, hidden_size_2, hidden_size_3, hidden_size_4).to(device)
    early_stopping = EarlyStopping(patience=10, delta=0.00, restore_best_weights=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss()

    model_results = training_loop(model=model,
                                  train_dataloader=train_dataloader_optim,
                                  validation_dataloader=validation_dataloader_optim,
                                  loss_fn=loss_fn,
                                  optimizer=optimizer,
                                  device=device,
                                  epochs=NUM_EPOCHS,
                                  early_stopping=early_stopping)

    print(f"Returning the best validation loss: {model_results['Validation_loss'][-10-1]} after {len(model_results['Validation_loss'])} epochs")
    return model_results["Validation_loss"][-10-1]

Running the hyperparameters optimization process

In [ ]:
study = optuna.create_study(direction='minimize')

study.optimize(hyperparameters_tuning, n_trials=50)

print("Best hyperparameters: ", study.best_params)
print("Best validation loss: ", study.best_value)

Saving the study

In [ ]:
import joblib

joblib.dump(study, '/content/drive/MyDrive/Colab Notebooks/My_Model/study.pkl')

Final training to get the final model using the best hyperparameters.

In [ ]:
import torch
from timeit import default_timer as timer

NUM_EPOCHS = 1000

best_learning_rate, best_batch_size, best_hidden_size_1, best_hidden_size_2, best_hidden_size_3, best_hidden_size_4 = study.best_params.values()

final_model = ModelArchitecture(best_hidden_size_1, best_hidden_size_2, best_hidden_size_3, best_hidden_size_4).to(device)
early_stopping = EarlyStopping(patience=20, delta=0.00, restore_best_weights=True)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=final_model.parameters(), lr=best_learning_rate)

start_time = timer()

train_dataloader = train_data_loader(train_dataset=train_data,
                                     batch_size=best_batch_size,
                                     shuffle=True,
                                     num_workers=NUM_WORKERS)

validation_dataloader = validation_data_loader(validation_dataset=validation_data,
    batch_size=best_batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS)

final_model_results = training_loop(model=final_model,
                                    train_dataloader=train_dataloader,
                                    validation_dataloader=validation_dataloader,
                                    loss_fn=loss_fn,
                                    optimizer=optimizer,
                                    device=device,
                                    epochs=NUM_EPOCHS,
                                    early_stopping=early_stopping)

end_time = timer()

print(f"Total training time: {end_time-start_time:.3f} seconds")

Saving the model weights

In [ ]:
model_weights_path = '/content/drive/MyDrive/Colab Notebooks/My_Model/No Data Augmentation/model_weights.pth'
torch.save(final_model.state_dict(), model_weights_path)

Loading the saved model weights

In [ ]:
best_learning_rate, best_batch_size, best_hidden_size_1, best_hidden_size_2, best_hidden_size_3, best_hidden_size_4 = 3694, 1093, 562, 374

loaded_model = ModelArchitecture(best_hidden_size_1, best_hidden_size_2, best_hidden_size_3, best_hidden_size_4).to(device)

loaded_model.load_state_dict(torch.load('/content/drive/MyDrive/Colab Notebooks/My_Model/No Data Augmentation/model_weights.pth', weights_only=True))

next(iter(loaded_model.parameters())).device

Defining a function to plot the loss curves

In [ ]:
import matplotlib.pyplot as plt

def plot_loss_curves(results: dict[str, list[float]]):

    loss = results['Train_loss']
    validation_loss = results['Validation_loss']

    accuracy = results['Train_accuracy']
    validation_accuracy = results['Validation_accuracy']

    epochs = range(len(results['Train_loss']))

    plt.figure(figsize=(15, 7))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, loss, label='Train_loss')
    plt.plot(epochs, validation_loss, label='Validation_loss')
    plt.axvline(x = len(epochs)-21, color = 'r', linestyle='--', label = 'Early Stopping')
    plt.title('Loss')
    plt.xlabel('Epochs')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, accuracy, label='Train_accuracy')
    plt.plot(epochs, validation_accuracy, label='Validation_accuracy')
    plt.axvline(x = len(epochs)-21, color = 'r', linestyle='--', label = 'Early Stopping')
    plt.title('Accuracy')
    plt.xlabel('Epochs')
    plt.legend();

Plotting the loss curves

In [ ]:
plot_loss_curves(final_model_results)

In [ ]:
!pip install torchmetrics

Defining the test step.

In [ ]:
from torchmetrics.classification import ConfusionMatrix, Recall, F1Score, Specificity, MulticlassPrecision

def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device=device):

  model.eval()

  test_loss = 0
  test_accuracy = 0
  predictions = []

  with torch.inference_mode():
    for batch, (X_test, y_test) in enumerate(dataloader):

      X_test, y_test = X_test.to(device), y_test.to(device)

      test_logits = model(X_test)

      loss = loss_fn(test_logits, y_test)
      test_loss += loss.item()

      test_probs = torch.softmax(test_logits, dim=1)
      test_preds = torch.argmax(test_probs, dim=1)
      predictions.append(test_preds.cpu())

      accuracy = (test_preds == y_test).sum().item()/len(test_preds)
      test_accuracy += accuracy

  test_loss /= len(dataloader)
  test_accuracy /= len(dataloader)

  predictions_tensor = torch.cat(predictions).to(device)
  targets_tensor = torch.tensor(test_data.targets).to(device)

  confmat = ConfusionMatrix(task="multiclass", num_classes=4).to(device)
  confusion_matrix = confmat(predictions_tensor, targets_tensor)

  precision = MulticlassPrecision(num_classes=4, average='macro').to(device)
  precision_result = precision(predictions_tensor, targets_tensor)

  recall = Recall(task='multiclass', num_classes=4).to(device)
  recall_result = recall(predictions_tensor, targets_tensor)

  specificity = Specificity(task='multiclass', num_classes=4).to(device)
  specificity_result = specificity(predictions_tensor, targets_tensor)

  f1_score = F1Score(task='multiclass', num_classes=4).to(device)
  f1_score_result = f1_score(predictions_tensor, targets_tensor)

  return round(test_loss,3), round(test_accuracy,3), round(recall_result.item(),3), round(f1_score_result.item(),3), round(specificity_result.item(),3), confusion_matrix, round(precision_result.item(), 3)

Retrieving the evaluation metrics of the final model in the test dataset.

In [ ]:
from torchmetrics.classification import ConfusionMatrix

model_loss, model_accuracy, model_recall, model_f1_score, model_specificity, model_confusion_matrix, model_precision = test_step(model=loaded_model,
                                                    dataloader=test_dataloader,
                                                    loss_fn=loss_fn,
                                                    device=device)

print(f'Loss: {model_loss:.3f}\nAccuracy: {model_accuracy:.3f}\nRecall: {model_recall:.3f}\nF1 Score: {model_f1_score:.3f}\nSpecificity: {model_specificity:.3f}\nPrecision: {model_precision:.3f}')

Plotting the Confusion Matrix

In [ ]:
from mlxtend.plotting import plot_confusion_matrix

fig, ax = plot_confusion_matrix(
    conf_mat=model_confusion_matrix.cpu().numpy(),
    class_names=train_data.classes,
    figsize=(10, 7)
);